# Terra Signal: variável de segmentação (Grupo 1)

Estudo de qual variável divide melhor a base e de se vale ter **modelos separados**, com a mesma fórmula e pesos diferentes em cada segmento.

Cada pergunta tem três células: **pergunta** → **código** → **resposta**.

- **Limpeza:** regras do notebook 06.
- **Dummies:** regras do notebook 07, com `tenure` em 6 faixas e sem `TotalCharges` (a pergunta 6 testa o efeito de incluí-la). Os cortes são recalculados no treino de cada fold.
- **Técnica de comparação:** regressão logística, escolhida porque seus pesos são diretos de comparar. Ela serve para testar a segmentação e não é a escolha final do modelo.
- `MonthlyIncome`, `MonthlyCharges` e `CustomerFeedback` não entram.
- Roda com pandas, numpy, scipy, statsmodels e scikit-learn, sem Spark. Funciona localmente e no Databricks (Git folder).

## Setup

In [1]:
%pip install -q statsmodels  # o ambiente serverless do Databricks pode não ter

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


In [2]:
import itertools
import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import chi2, chi2_contingency
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.tree import DecisionTreeClassifier, export_text

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

DATA_DIR = "../data/raw/"  # este notebook fica em notebooks/; os CSVs ficam em data/raw/
hist_raw = pd.read_csv(DATA_DIR + "history.csv")
inf_raw = pd.read_csv(DATA_DIR + "inference.csv")
print("history:", hist_raw.shape)

history: (5634, 23)


In [3]:
def prep(d):
    """Regras do notebook 06: padroniza PhoneService, recupera tenure e imputa TotalCharges."""
    d = d.copy()
    d["PhoneService"] = d["PhoneService"].str.capitalize()
    d["tenure_num"] = pd.to_numeric(d["tenure"], errors="coerce")
    d["tenure_num"] = d["tenure_num"].fillna((d["TotalCharges"] / d["MonthlyCharges"]).round())
    d["TotalCharges_imp"] = d["TotalCharges"].fillna(d["MonthlyCharges"] * d["tenure_num"])
    d.loc[d["tenure_num"] == 0, "TotalCharges_imp"] = 0
    return d


hist = prep(hist_raw)
hist = hist[hist["tenure_num"].notna()].reset_index(drop=True)  # 1 linha sem tenure e sem TotalCharges
inf = prep(inf_raw)
y = (hist["Churn"] == "Yes").astype(int)
print("Clientes no treino:", len(hist), "| churners:", y.sum(), f"({y.mean():.1%})")

Clientes no treino: 5633 | churners: 1503 (26.7%)


In [4]:
def melhor_corte(x, y, min_folha):
    """Corte binário de maior qui-quadrado, com p-valor corrigido por Bonferroni pelo nº de cortes testados."""
    tab = pd.DataFrame({"x": x, "y": y}).groupby("x")["y"].agg(["size", "sum"]).sort_index()
    n_esq, c_esq = tab["size"].cumsum().values, tab["sum"].cumsum().values
    n, c = n_esq[-1], c_esq[-1]
    validos = (n_esq >= min_folha) & (n - n_esq >= min_folha)
    if not validos.any():
        return None
    a, b = c_esq, n_esq - c_esq
    e, f = c - c_esq, (n - n_esq) - (c - c_esq)
    den = (a + b) * (e + f) * (a + e) * (b + f)
    with np.errstate(divide="ignore", invalid="ignore"):
        q = np.where(den > 0, n * (a * f - b * e) ** 2 / den, 0.0)
    q = np.where(validos, q, -1.0)
    i = int(np.argmax(q))
    return float(tab.index[i]), float(q[i]), min(1.0, chi2.sf(q[i], 1) * int(validos.sum()))


def aid(x, y, min_folha=0.05, alfa=0.05, max_faixas=None):
    """Divide sempre a faixa com maior qui-quadrado, até faltar significância, tamanho ou o limite de faixas."""
    x, y = np.asarray(x, float), np.asarray(y)
    tam = int(np.ceil(min_folha * len(x)))
    nos, passos = [np.ones(len(x), bool)], []
    while not max_faixas or len(nos) < max_faixas:
        candidatos = []
        for k, m in enumerate(nos):
            r = melhor_corte(x[m], y[m], tam)
            if r and r[2] < alfa:
                candidatos.append((r[1], k, r))
        if not candidatos:
            break
        _, k, r = max(candidatos, key=lambda z: z[0])
        m = nos.pop(k)
        nos += [m & (x <= r[0]), m & (x > r[0])]
        passos.append({"corte": r[0], "qui2": round(r[1], 1), "p_bonferroni": r[2]})
    return sorted(p["corte"] for p in passos), passos


def faixas(x, cortes):
    return pd.cut(x, [-np.inf] + list(cortes) + [np.inf])



DUMMIES_CATEG = {
    "gender": ["Female"],
    "SeniorCitizen": [1],
    "Partner": ["Yes"],
    "Dependents": ["Yes"],
    "MultipleLines": ["Yes", "No phone service"],
    "InternetService": ["DSL", "No"],
    "OnlineSecurity": ["Yes"],
    "OnlineBackup": ["Yes"],
    "DeviceProtection": ["Yes"],
    "TechSupport": ["Yes"],
    "StreamingTV": ["Yes"],
    "StreamingMovies": ["Yes"],
    "Contract": ["One year", "Two year"],
    "PaperlessBilling": ["No"],
    "PaymentMethod": ["Mailed check", "Credit card (automatic)", "Bank transfer (automatic)"],
}


def dummies_categoricas(d):
    """Uma coluna 0/1 para cada categoria listada. A categoria omitida é a referência."""
    return pd.DataFrame(
        {f"{col}={val}": (d[col] == val).astype(int) for col, vals in DUMMIES_CATEG.items() for val in vals},
        index=d.index,
    )



def dummies_faixas(x, cortes, nome, ref):
    """Uma dummy por faixa, exceto a faixa de referência `ref` (posição da faixa)."""
    f = np.digitize(x, cortes, right=True)
    limites = [-np.inf] + list(cortes) + [np.inf]
    return pd.DataFrame(
        {f"{nome}=({limites[i]:g}, {limites[i + 1]:g}]": (f == i).astype(int) for i in range(len(limites) - 1) if i != ref},
        index=x.index,
    )


def faixa_referencia(x, cortes):
    """A referência é a faixa mais frequente."""
    return int(pd.Series(np.digitize(x, cortes, right=True)).value_counts().idxmax())

## Como a segmentação é avaliada

Uma variável boa para segmentar precisa passar por três filtros, nesta ordem:

1. **Separar bem:** usamos o AID na raiz, com o qui-quadrado da melhor divisão em dois grupos, e a primeira divisão de uma árvore.
2. **Ter churners suficientes:** cada segmento precisa de eventos para estimar seus próprios pesos.
3. **Melhorar o modelo:** forte ligação com churn não basta, porque uma dummy já captura a diferença de nível entre os grupos. Equações separadas só compensam se melhorarem a previsão **fora da amostra**.

### 1. Pelo AID, qual variável separa melhor a base em dois grupos?

In [5]:
CATEGORICAS = ["Contract", "OnlineSecurity", "TechSupport", "InternetService", "PaymentMethod", "OnlineBackup",
               "DeviceProtection", "StreamingMovies", "StreamingTV", "PaperlessBilling", "Dependents", "SeniorCitizen",
               "Partner", "MultipleLines", "gender"]
MIN_GRUPO = int(np.ceil(0.05 * len(hist)))
linhas = []
for c in CATEGORICAS:
    cats = hist[c].unique().tolist()
    melhor = None
    for r in range(1, len(cats) // 2 + 1):
        for g in itertools.combinations(cats, r):
            m = hist[c].isin(g)
            if min(m.sum(), (~m).sum()) < MIN_GRUPO:
                continue
            q = chi2_contingency(pd.crosstab(m, y), correction=False)[0]
            if melhor is None or q > melhor[0]:
                melhor = (q, m, " + ".join(map(str, g)))
    linhas.append((c, melhor[0], melhor[2], melhor[1]))
for v in ["tenure_num", "TotalCharges_imp"]:
    corte, q, _ = melhor_corte(hist[v].values, y.values, MIN_GRUPO)
    linhas.append((v, q, f"até {corte:g}", hist[v] <= corte))

ranking = pd.DataFrame([{
    "variavel": v, "qui2": round(q, 1), "grupo_A": g, "n_A": int(m.sum()), "churners_A": int(y[m].sum()), "taxa_A": round(y[m].mean(), 3),
    "n_B": int((~m).sum()), "churners_B": int(y[~m].sum()), "taxa_B": round(y[~m].mean(), 3),
} for v, q, g, m in linhas]).sort_values("qui2", ascending=False).reset_index(drop=True)
GRUPOS = {v: m.values for v, q, g, m in linhas}
print(ranking.to_string())

            variavel   qui2              grupo_A   n_A  churners_A  taxa_A   n_B  churners_B  taxa_B
0           Contract  898.9       Month-to-month  3099        1322   0.427  2534         181   0.071
1     OnlineSecurity  662.2                   No  2803        1175   0.419  2830         328   0.116
2        TechSupport  648.2                   No  2790        1167   0.418  2843         336   0.118
3         tenure_num  552.0               até 17  2115         942   0.445  3518         561   0.159
4    InternetService  532.9          Fiber optic  2476        1041   0.420  3157         462   0.146
5      PaymentMethod  493.9     Electronic check  1888         852   0.451  3745         651   0.174
6       OnlineBackup  390.8                   No  2457         981   0.399  3176         522   0.164
7   DeviceProtection  341.1                   No  2469         963   0.390  3164         540   0.171
8    StreamingMovies  305.7  No internet service  1233          89   0.072  4400        141

**Resposta:** **`Contract`**, separando contrato mensal dos demais (qui-quadrado de 899): 42,7% de churn contra 7,1%. Em seguida vêm `OnlineSecurity` (662), `TechSupport` (648), `tenure` até 17 meses (552), `InternetService` (533) e `PaymentMethod` (494). Como todas as divisões têm dois grupos e usam a mesma base, os valores podem ser comparados diretamente.

### 2. O que uma árvore de decisão divide primeiro?

In [6]:
X_arv = pd.get_dummies(hist[CATEGORICAS].astype(str)).astype(int)
X_arv["tenure"] = hist["tenure_num"]
X_arv["TotalCharges"] = hist["TotalCharges_imp"]
arvore = DecisionTreeClassifier(max_depth=3, min_samples_leaf=MIN_GRUPO, random_state=0).fit(X_arv, y)
print(export_text(arvore, feature_names=list(X_arv.columns), show_weights=True))

rng = np.random.RandomState(0)
raizes = []
for _ in range(200):
    i = rng.randint(0, len(hist), len(hist))
    t = DecisionTreeClassifier(max_depth=1, min_samples_leaf=MIN_GRUPO, random_state=0).fit(X_arv.iloc[i], y.iloc[i])
    raizes.append(X_arv.columns[t.tree_.feature[0]])
print("Primeira divisão em 200 reamostragens:", pd.Series(raizes).value_counts().to_dict())

|--- Contract_Month-to-month <= 0.50
|   |--- InternetService_Fiber optic <= 0.50
|   |   |--- OnlineSecurity_No <= 0.50
|   |   |   |--- weights: [1414.00, 37.00] class: 0
|   |   |--- OnlineSecurity_No >  0.50
|   |   |   |--- weights: [274.00, 31.00] class: 0
|   |--- InternetService_Fiber optic >  0.50
|   |   |--- Contract_One year <= 0.50
|   |   |   |--- weights: [314.00, 28.00] class: 0
|   |   |--- Contract_One year >  0.50
|   |   |   |--- weights: [351.00, 85.00] class: 0
|--- Contract_Month-to-month >  0.50
|   |--- InternetService_Fiber optic <= 0.50
|   |   |--- tenure <= 4.50
|   |   |   |--- weights: [309.00, 235.00] class: 0
|   |   |--- tenure >  4.50
|   |   |   |--- weights: [698.00, 159.00] class: 0
|   |--- InternetService_Fiber optic >  0.50
|   |   |--- tenure <= 14.50
|   |   |   |--- weights: [245.00, 547.00] class: 1
|   |   |--- tenure >  14.50
|   |   |   |--- weights: [525.00, 381.00] class: 0



Primeira divisão em 200 reamostragens: {'Contract_Month-to-month': 200}


**Resposta:** a árvore confirma o AID: a primeira divisão é **contrato mensal**, em 200 de 200 reamostragens. Depois vem fibra, nos dois lados. A partir daí, os caminhos mudam:

| Contrato | Internet | Terceira divisão | Churn nas folhas |
|---|---|---|---|
| Mensal | Fibra | `tenure` até 14 meses | 69,1% e 42,1% |
| Mensal | Outras | `tenure` até 4 meses | 43,2% e 18,6% |
| Demais | Fibra | Um ano ou dois anos | 19,5% e 8,2% |
| Demais | Outras | Sem ou com `OnlineSecurity` | 10,2% e 2,6% |

A árvore também encontra cortes de `tenure` próximos aos do AID (4,5 e 14,5, contra 5 e 17), usando outro critério. O fato de `tenure` aparecer só no ramo mensal sugere um comportamento diferente entre os grupos, e a pergunta 4 testa se isso melhora a previsão.

### 3. Os segmentos têm churners suficientes?

Avaliamos as 6 variáveis de maior qui-quadrado, cada uma com sua melhor divisão em dois grupos. As dummies do modelo final são 25, então a coluna `churners_por_dummy` mostra o volume de eventos para estimar os pesos em cada segmento.

In [7]:
CANDIDATAS = ranking["variavel"].head(6).tolist()
N_DUMMIES = 25
print(pd.DataFrame([{
    "variavel": v, "grupo_A": ranking.set_index("variavel").loc[v, "grupo_A"],
    "churners_A": int(y[GRUPOS[v]].sum()), "churners_B": int(y[~GRUPOS[v]].sum()),
    "churners_por_dummy_no_menor": round(min(y[GRUPOS[v]].sum(), y[~GRUPOS[v]].sum()) / N_DUMMIES, 1),
} for v in CANDIDATAS]).to_string(index=False))
print("\nContract em três grupos:", y.groupby(hist["Contract"]).sum().to_dict())

       variavel          grupo_A  churners_A  churners_B  churners_por_dummy_no_menor
       Contract   Month-to-month        1322         181                          7.2
 OnlineSecurity               No        1175         328                         13.1
    TechSupport               No        1167         336                         13.4
     tenure_num           até 17         942         561                         22.4
InternetService      Fiber optic        1041         462                         18.5
  PaymentMethod Electronic check         852         651                         26.0

Contract em três grupos: {'Month-to-month': 1322, 'One year': 137, 'Two year': 44}


**Resposta:** a melhor variável é justamente a mais limitada. No grupo sem contrato mensal, `Contract` tem só 181 churners, cerca de 7 por dummy, abaixo da regra prática comum de 10 eventos por variável. Separar em três grupos seria ainda pior: 137 churners em um ano e 44 em dois anos. As outras cinco candidatas ficam entre 13 e 26 churners por dummy no menor grupo.

### 4. Modelos separados preveem melhor que um modelo único?

Comparamos, nos mesmos folds (5 folds, 4 repetições):

- **Modelo único:** todas as dummies, um só conjunto de pesos. As dummies já permitem um nível diferente para cada grupo.
- **Modelos separados:** a mesma fórmula, ajustada em cada segmento. A probabilidade de cada cliente vem do modelo do seu segmento.

Valores positivos em `dif_AUC` favorecem os modelos separados. Em `dif_logloss`, valores **negativos** é que favorecem.

In [8]:
CV = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=0)
X_cat = dummies_categoricas(hist)
FOLDS = list(CV.split(hist, y))


def preparar_folds(com_totalcharges=False):
    saida = []
    for tr, te in FOLDS:
        pa, pb = [X_cat.iloc[tr]], [X_cat.iloc[te]]
        config = [("tenure_num", "tenure", 6)] + ([("TotalCharges_imp", "tc", None)] if com_totalcharges else [])
        for v, nome, k in config:
            cortes = aid(hist[v].iloc[tr], y.iloc[tr], max_faixas=k)[0]
            ref = faixa_referencia(hist[v].iloc[tr], cortes)
            pa.append(dummies_faixas(hist[v].iloc[tr], cortes, nome, ref))
            pb.append(dummies_faixas(hist[v].iloc[te], cortes, nome, ref))
        saida.append((tr, te, pd.concat(pa, axis=1), pd.concat(pb, axis=1)))
    return saida


def comparar(seg, folds):
    r = {k: [] for k in ["u", "s", "lu", "ls", "uA", "sA", "uB", "sB"]}
    for tr, te, Xtr, Xte in folds:
        ytr, yte = y.iloc[tr], y.iloc[te]
        str_, ste = seg[tr], seg[te]
        pu = LogisticRegression(max_iter=3000).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
        ps = np.zeros(len(te))
        for val in [True, False]:
            ps[ste == val] = LogisticRegression(max_iter=3000).fit(Xtr[str_ == val], ytr[str_ == val]).predict_proba(Xte[ste == val])[:, 1]
        r["u"].append(roc_auc_score(yte, pu)); r["s"].append(roc_auc_score(yte, ps))
        r["lu"].append(log_loss(yte, pu)); r["ls"].append(log_loss(yte, ps))
        r["uA"].append(roc_auc_score(yte[ste], pu[ste])); r["sA"].append(roc_auc_score(yte[ste], ps[ste]))
        r["uB"].append(roc_auc_score(yte[~ste], pu[~ste])); r["sB"].append(roc_auc_score(yte[~ste], ps[~ste]))
    r = {k: np.array(v) for k, v in r.items()}
    return {
        "AUC_unico": round(r["u"].mean(), 4), "AUC_separados": round(r["s"].mean(), 4),
        "dif_AUC": round(np.mean(r["s"] - r["u"]), 4), "dp_dif": round(np.std(r["s"] - r["u"]), 4),
        "dif_logloss": round(np.mean(r["ls"] - r["lu"]), 4), "dp_logloss": round(np.std(r["ls"] - r["lu"]), 4),
        "A_unico": round(r["uA"].mean(), 3), "A_separado": round(r["sA"].mean(), 3),
        "B_unico": round(r["uB"].mean(), 3), "B_separado": round(r["sB"].mean(), 3),
    }


folds_base = preparar_folds()
print(pd.DataFrame([{"variavel": v, **comparar(GRUPOS[v], folds_base)} for v in CANDIDATAS]).to_string(index=False))

       variavel  AUC_unico  AUC_separados  dif_AUC  dp_dif  dif_logloss  dp_logloss  A_unico  A_separado  B_unico  B_separado
       Contract     0.8448         0.8443  -0.0005  0.0024       0.0010      0.0030    0.759       0.757    0.776       0.773
 OnlineSecurity     0.8448         0.8435  -0.0013  0.0018       0.0013      0.0022    0.784       0.783    0.821       0.819
    TechSupport     0.8448         0.8432  -0.0017  0.0017       0.0017      0.0018    0.785       0.784    0.819       0.816
     tenure_num     0.8448         0.8436  -0.0012  0.0014       0.0013      0.0018    0.795       0.793    0.823       0.822
InternetService     0.8448         0.8435  -0.0013  0.0018       0.0018      0.0019    0.789       0.787    0.833       0.832
  PaymentMethod     0.8448         0.8435  -0.0013  0.0022       0.0019      0.0023    0.773       0.770    0.840       0.839


**Resposta:** **não.** Em nenhuma das 6 candidatas os modelos separados superam o modelo único. A AUC fica sempre um pouco menor (de −0,0005 a −0,0017), e a log-loss fica sempre um pouco pior. Por outro lado, as diferenças estão dentro de cerca de um desvio-padrão, então a leitura correta é que a segmentação não traz ganho, e não que ela prejudica de forma clara.

No caso de `Contract`, a AUC dentro dos segmentos também não melhora: 0,759 → 0,757 no mensal e 0,776 → 0,773 nos demais.

### 5. Os pesos mudam entre contrato mensal e os demais?

In [9]:
SEG = GRUPOS["Contract"]
CORTES = aid(hist["tenure_num"], y, max_faixas=6)[0]
X = pd.concat([X_cat, dummies_faixas(hist["tenure_num"], CORTES, "tenure", faixa_referencia(hist["tenure_num"], CORTES))], axis=1).astype(float)

print("Churners por faixa de tenure, nos dois segmentos:")
print(pd.crosstab(faixas(hist["tenure_num"], CORTES), SEG, values=y, aggfunc="sum").rename(columns={True: "mensal", False: "demais"}).rename_axis(columns=None).to_string())


def ajuste(mascara, retirar=()):
    cols = [c for c in X.columns[X[mascara].std() > 0] if c not in retirar]
    return sm.Logit(y[mascara], sm.add_constant(X.loc[mascara, cols])).fit(disp=0, maxiter=500)


# Sem contrato mensal, as duas dummies de Contract somam 1: uma delas precisa sair para o ajuste ser identificável.
unico, mensal, demais = ajuste(np.ones(len(y), bool)), ajuste(SEG), ajuste(~SEG, retirar=["Contract=Two year"])
lr = 2 * (mensal.llf + demais.llf - unico.llf)
gl = len(mensal.params) + len(demais.params) - len(unico.params)
print(f"\nTeste de razão de verossimilhança: LR = {lr:.1f}, graus de liberdade = {gl}, p = {chi2.sf(lr, gl):.4f}")

pesos = pd.DataFrame({
    "OR_unico": np.exp(unico.params), "OR_mensal": np.exp(mensal.params), "p_mensal": mensal.pvalues,
    "OR_demais": np.exp(demais.params), "p_demais": demais.pvalues,
}).drop(["const", "Contract=One year", "Contract=Two year"]).round(3)  # Contract é constante ou muda de referência nos segmentos
print(pesos.to_string())

Churners por faixa de tenure, nos dois segmentos:
              demais  mensal
tenure_num                  
(-inf, 5.0]        4     587
(5.0, 17.0]       12     339
(17.0, 43.0]      47     306
(43.0, 59.0]      59      67
(59.0, 70.0]      49      22
(70.0, inf]       10       1



Teste de razão de verossimilhança: LR = 38.4, graus de liberdade = 23, p = 0.0231
                                         OR_unico  OR_mensal  p_mensal  OR_demais  p_demais
Dependents=Yes                              0.857      0.870     0.236      0.834     0.355
DeviceProtection=Yes                        1.032      1.136     0.211      0.798     0.223
InternetService=DSL                         0.404      0.386     0.000      0.521     0.003
InternetService=No                          0.170      0.182     0.000      0.224     0.001
MultipleLines=No phone service              1.594      1.761     0.000      1.146     0.689
MultipleLines=Yes                           1.386      1.430     0.000      1.247     0.258
OnlineBackup=Yes                            0.889      0.825     0.048      1.161     0.418
OnlineSecurity=Yes                          0.719      0.748     0.010      0.661     0.022
PaperlessBilling=No                         0.746      0.739     0.001      0.805     0.2

**Resposta:** alguns pesos parecem mudar, mas com pouca precisão no grupo menor. O teste conjunto indica diferença (LR = 38,4, com 23 graus de liberdade, p = 0,023). As maiores diferenças estão em:

| Variável | OR no mensal | OR nos demais |
|---|---|---|
| `SeniorCitizen=1` | 1,41 | 0,87 |
| `StreamingMovies=Yes` | 1,18 | 2,00 |
| `OnlineBackup=Yes` | 0,83 | 1,16 |
| `TechSupport=Yes` | 0,64 | 0,90 |

Isso combina com a pergunta 4: os pesos podem até diferir, mas os dados do grupo menor não bastam para transformar isso em previsão melhor. Nas faixas de até 5 e de 5 a 17 meses, por exemplo, esse grupo tem só 4 e 12 churners. As comparações individuais da tabela não têm correção para múltiplos testes.

### 6. O resultado muda se `TotalCharges` entrar nas dummies?

In [10]:
folds_tc = preparar_folds(com_totalcharges=True)
print(pd.DataFrame([{"variavel": "Contract", **comparar(GRUPOS["Contract"], folds_tc)}]).to_string(index=False))

variavel  AUC_unico  AUC_separados  dif_AUC  dp_dif  dif_logloss  dp_logloss  A_unico  A_separado  B_unico  B_separado
Contract     0.8454         0.8447  -0.0007  0.0025       0.0012      0.0031     0.76       0.759    0.778       0.773


**Resposta:** não. Com `TotalCharges`, a conclusão se mantém para `Contract`: a diferença entre modelos separados e modelo único é de −0,0007 ± 0,0025.

## Decisão

**Usar um modelo único, com `Contract` como dummy, sem segmentar a base.** `Contract` é claramente a variável que mais separa a base, pelo AID e pela árvore, mas equações separadas não melhoram a previsão. Um modelo único é mais simples e usa todos os dados para estimar cada peso.

## Limites desta análise

- **Só divisões em dois grupos:** não testamos três ou mais segmentos, nem combinações como contrato mensal com fibra, que deixariam os grupos menores ainda menores.
- **Os cortes de `tenure` são os mesmos nos dois segmentos:** recalculá-los por segmento é outra forma de segmentar, que não foi testada.
- **A conclusão vale para a regressão logística:** ela usa a mesma regularização nos dois lados, e outra técnica pode se comportar de forma diferente.
- **A AUC mede ordenação:** a log-loss também não favoreceu a segmentação, mas o efeito sobre o ponto de corte financeiro não foi avaliado.
- **Ao longo do tempo:** o histórico não tem data, então não dá para testar se os pesos de cada segmento permanecem estáveis.